# Week 5 ablations, session b: kl0 on GPU 0, kl001 on GPU 1


In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "math-verify[antlr4_13_2]" peft 2>&1 | tail -1
# Kaggle's preinstalled torchao (0.10) is too old for peft, which imports it if present; we don't use it.
!pip uninstall -y -q torchao
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"
!python -m pytest -q tests/ 2>&1 | tail -2

In [ ]:
RUNS = [["--config", "configs/abl-kl0.yaml", "--set", "out_dir=/kaggle/working/checkpoints"], ["--config", "configs/abl-kl001.yaml", "--set", "out_dir=/kaggle/working/checkpoints"]]
env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
procs = []
for gpu, args in enumerate(RUNS):
    path = f"/kaggle/working/train-{gpu}.log"
    p = subprocess.Popen(["python", "-m", "grpo.train", *args], env={**env, "CUDA_VISIBLE_DEVICES": str(gpu)},
                         stdout=open(path, "w"), stderr=subprocess.STDOUT)
    procs.append((args, path, p))
for args, path, p in procs:
    print(" ".join(args), "-> exit code", p.wait())
    print("".join(l for l in open(path) if l.startswith(("step", "resumed", "trainable", "non-finite")) or "Error" in l)[-8000:])
